In [67]:
import sys
import numpy as np
import pandas as pd

from scipy.stats import jarque_bera
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller

sys.path.append('..')
from modules.source import Sources, parquet_daily, parquet_monthly

DOMESTIC = [Sources.crude_palm_oil, Sources.brent_oil, Sources.cpi_inflation]
GLOBAL = [Sources.rate_differential, Sources.UST_10Y, Sources.VIX, Sources.DXY]

df = pd.read_parquet(parquet_monthly)
change = df.xs("change", axis=1, level=1)

d = change[[Sources.KLCI] + DOMESTIC + GLOBAL].dropna()
n = len(d)

print(f"n = {n} months  ({d.index.min().strftime('%Y-%m-%d')} to {d.index.max().strftime('%Y-%m-%d')})")

def fit_hac(rhs_vars):
  formula = f"{Sources.KLCI} ~ {' + '.join(rhs_vars)}"
  return smf.ols(formula, data=d).fit(cov_type='HAC', cov_kwds={'maxlags': int(np.floor(4 * (n / 100) ** (2 / 9)))})

m_dom = fit_hac(DOMESTIC)
m_global = fit_hac(GLOBAL)
m_full = fit_hac(DOMESTIC + GLOBAL)

n = 139 months  (2015-02-28 to 2026-08-31)


### 1. ADF (Stationary), VIF (Multicollinearity) and Jarque-Bera (Normality)
Verify the statistical assumptions so the outputs aren't producing spurious correlations.

In [68]:
print('1. Augmented Dickey-Fuller (ADF)')

for col in [Sources.KLCI] + DOMESTIC + GLOBAL:
  stat, p = adfuller(d[col], autolag='AIC', result_object=True)
  status = 'Pass (I(0))' if p < 0.05 else 'FAIL (Non-Stationary)'
  print(f'  {col:16s} ADF = {stat:7.3f} | p = {p:.4f} | {status}')

# A. Multicollinearity Check (VIF)
X = m_full.model.exog
vif = pd.Series(
  [variance_inflation_factor(X, i) for i in range(X.shape[1])],
  index=m_full.model.exog_names,
).drop('Intercept')

print('\n2. Variance Inflation Factor (VIF):')
print(vif.round(2))

# B. Residual Normality Check (Jarque-Bera)
jb_stat, jb_p = jarque_bera(m_full.resid)
print(f'\n3. Residual Normality (Jarque-Bera): Stat = {jb_stat:.3f}, p = {jb_p:.4f}')

1. Augmented Dickey-Fuller (ADF)
  KLCI             ADF = -13.147 | p = 0.0000 | Pass (I(0))
  CPO              ADF =  -9.142 | p = 0.0000 | Pass (I(0))
  Oil              ADF =  -8.587 | p = 0.0000 | Pass (I(0))
  cpi_inflation    ADF =  -9.356 | p = 0.0000 | Pass (I(0))
  FFR_OPR_diff     ADF =  -4.740 | p = 0.0001 | Pass (I(0))
  UST_10Y          ADF = -10.891 | p = 0.0000 | Pass (I(0))
  VIX              ADF =  -5.633 | p = 0.0000 | Pass (I(0))
  DXY              ADF = -11.484 | p = 0.0000 | Pass (I(0))

2. Variance Inflation Factor (VIF):
CPO              1.04
Oil              1.45
cpi_inflation    1.07
FFR_OPR_diff     1.14
UST_10Y          1.38
VIX              1.13
DXY              1.25
dtype: float64

3. Residual Normality (Jarque-Bera): Stat = 0.066, p = 0.9673


### 4. MODEL FIT & VARIANCE DECOMPOSITION

In [69]:
r2_df = pd.DataFrame(
   {
      'R2': [m_dom.rsquared, m_global.rsquared, m_full.rsquared],
      "Adj R2": [m_dom.rsquared_adj, m_global.rsquared_adj, m_full.rsquared_adj],
   },
   index=['Domestic only', 'Fed only', 'Domestic + Fed'],
)
print(r2_df.round(6))

incr_global = m_full.rsquared - m_dom.rsquared
incr_dom = m_full.rsquared - m_global.rsquared
overlap = m_dom.rsquared + m_global.rsquared - m_full.rsquared

print(f"\nFed's unique share of variance (Domestic -> +Fed):   {incr_global:.4f}")
print(f"Domestic's unique share of variance (Fed -> +Domestic): {incr_dom:.4f}")
print(f'Shared / overlapping variance:                         {overlap:.4f}\n')

hypothesis_dom = ", ".join(f"{v} = 0" for v in DOMESTIC)
wald_dom = m_full.wald_test(hypothesis_dom, use_f=True, scalar=True)

# Wald test for Fed Block
hypothesis_global = ", ".join(f"{v} = 0" for v in GLOBAL)
wald_global = m_full.wald_test(hypothesis_global, use_f=True, scalar=True)

print(f"Domestic Block Wald: F({wald_dom.df_num:.0f}, {wald_dom.df_denom:.0f}) = {wald_dom.statistic:.3f}, p = {wald_dom.pvalue:.4f}")
print(f"Fed Block Wald:      F({wald_global.df_num:.0f}, {wald_global.df_denom:.0f}) = {wald_global.statistic:.3f}, p = {wald_global.pvalue:.4f}")

                      R2    Adj R2
Domestic only   0.116779  0.097152
Fed only        0.293432  0.272340
Domestic + Fed  0.330541  0.294769

Fed's unique share of variance (Domestic -> +Fed):   0.2138
Domestic's unique share of variance (Fed -> +Domestic): 0.0371
Shared / overlapping variance:                         0.0797

Domestic Block Wald: F(3, 131) = 4.682, p = 0.0039
Fed Block Wald:      F(4, 131) = 12.462, p = 0.0000


### 5. INFERENCE & FEATURE IMPORTANCE

In [70]:
print('\n' + '=' * 65)
print('SECTION 5: FULL-MODEL INFERENCE & FEATURE IMPORTANCE')
print('=' * 65)

coefs = pd.DataFrame(
   {'coef': m_full.params, 'HAC SE': m_full.bse, 'p': m_full.pvalues}
).drop('Intercept')

# Vectorized Standardized Beta Calculation
coefs['std_beta'] = coefs['coef'] * d[coefs.index].std() / d[Sources.KLCI].std()

# Significance Flags
coefs['sig'] = coefs['p'].apply(lambda p: ('***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''))
print(coefs.round(4))


SECTION 5: FULL-MODEL INFERENCE & FEATURE IMPORTANCE
                 coef  HAC SE       p  std_beta  sig
CPO           -0.0122  0.0326  0.7084   -0.0304     
Oil            0.0352  0.0199  0.0771    0.1494    *
cpi_inflation -0.7728  0.4490  0.0852   -0.1188    *
FFR_OPR_diff  -0.0055  0.0142  0.6981   -0.0357     
UST_10Y        0.0125  0.0108  0.2465    0.0935     
VIX           -0.0016  0.0004  0.0001   -0.2949  ***
DXY           -0.5543  0.1305  0.0000   -0.3501  ***


In [71]:
print('\n' + '=' * 65)
print('SECTION 6: PER-VARIABLE PARTIAL R² (DROP-ONE)')
print('=' * 65)
print("Each variable's unique contribution: full-model R² minus the R²")
print('of the same model with that one variable removed, holding every')
print('other variable (domestic and Fed) fixed.\n')

all_vars = DOMESTIC + GLOBAL
partial_r2 = {}
for v in all_vars:
   remaining = [x for x in all_vars if x != v]
   m_minus = fit_hac(remaining)
   partial_r2[v] = m_full.rsquared - m_minus.rsquared

partial_r2 = pd.Series(partial_r2, name='partial_R2').sort_values(ascending=False)
print(partial_r2.round(4))

unassigned = m_full.rsquared - partial_r2.sum()
print(f"\nSum of partial R2s:            {partial_r2.sum():.4f}")
print(f"Full model R2:                 {m_full.rsquared:.4f}")
print(f"Unassigned (shared) variance:  {unassigned:.4f}")
print('(shared variance = movement multiple variables jointly explain,')
print(' credited to none of them individually when dropped one at a time)')


SECTION 6: PER-VARIABLE PARTIAL R² (DROP-ONE)
Each variable's unique contribution: full-model R² minus the R²
of the same model with that one variable removed, holding every
other variable (domestic and Fed) fixed.

DXY              0.0981
VIX              0.0767
Oil              0.0154
cpi_inflation    0.0132
UST_10Y          0.0064
FFR_OPR_diff     0.0011
CPO              0.0009
Name: partial_R2, dtype: float64

Sum of partial R2s:            0.2119
Full model R2:                 0.3305
Unassigned (shared) variance:  0.1187
(shared variance = movement multiple variables jointly explain,
 credited to none of them individually when dropped one at a time)
